# AAROH - Phase 4: Quiz Generation & Student Assessment Engine

This notebook demonstrates the complete Phase 4 workflow:
1. **Culturally-Localized Quiz Generation**: Generates 5 questions (MCQ + Fill-in-Blank + True/False) rooted in local village context.
2. **Student Submission & Grading**: Automated flexible grading (case-insensitive, synonyms).
3. **Concept-Level Gap Analysis**: Pinpoints gaps by Bloom's taxonomy level (Remember, Understand, Apply).
4. **Remedial Lesson Triggering**: Automatically attaches a tailored remedial lesson when score < 60%.
5. **Teacher Dashboard Insights**: Aggregates class-level performance and generates actionable recommendations.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.aaroh.pipeline.quiz_assessment_pipeline import QuizAssessmentPipeline

pipeline = QuizAssessmentPipeline()
print('Phase 4 Quiz & Assessment Pipeline initialized successfully!')

## 1. Generate Culturally-Localized Quiz for a Concept

In [ ]:
quiz = pipeline.generate_quiz_for_concept(
    concept_code='EVS-G3-WAT-01',
    target_language='hi',
    target_dialect='gon', # Gondi dialect context
    grade_level=3,
)

print(f'Quiz ID: {quiz.quiz_id}')
print(f'Concept: {quiz.concept_name} (Code: {quiz.concept_code})')
print(f'Total Questions: {len(quiz.questions)}\n')

for i, q in enumerate(quiz.questions, 1):
    print(f'Q{i} [{q.question_type.upper()} | Bloom: {q.blooms_level} | Diff: {q.difficulty_level}]:')
    print(f'   {q.question_text}')
    if q.options:
        for opt in q.options:
            print(f'      {opt}')
    print(f'   -> Correct Answer: {q.correct_answer}')
    print(f'   -> Explanation: {q.explanation}\n')

## 2. Simulate Student Attempt (Low Score -> Triggers Remedial Lesson)

In [ ]:
# Student gets Q1 wrong, Q2 wrong, Q3 right, Q4 wrong, Q5 wrong -> Score: 20%
student_answers = {
    'Q1': 'A',            # Wrong
    'Q2': 'A',            # Wrong
    'Q3': 'B',            # Correct
    'Q4': 'khatam hona',  # Wrong
    'Q5': 'True',         # Wrong
}

assessment_result = pipeline.submit_and_assess_quiz(
    quiz_id=quiz.quiz_id,
    student_id='student_ramu_01',
    answers=student_answers,
)

print(f"Score: {assessment_result['score_pct']}%")
print(f"Needs Remedial: {assessment_result['needs_remedial']}")
print(f"\nTeacher Insight:\n{assessment_result['teacher_insight']}")
print(f"\nLearning Gaps Identified: {len(assessment_result['learning_gaps'])}")
for gap in assessment_result['learning_gaps']:
    print(f"  - Bloom Level: {gap['blooms_level']} (Severity: {gap['gap_severity']})")

if assessment_result['remedial_lesson']:
    print("\n--- Attached Remedial Lesson ---")
    print("Simplified Story:", assessment_result['remedial_lesson']['story_narrative'][:200], '...')
    print("Hands-On Activity:", assessment_result['remedial_lesson']['hands_on_activity'].get('title'))

## 3. Teacher Dashboard & Class Analytics Report

In [ ]:
class_report = pipeline.get_class_report('EVS-G3-WAT-01')
print(json.dumps(class_report, indent=2, ensure_ascii=False))